In [1]:
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O
import os
import random
from random import random as rd
import gc
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor  # faster GBM
from sklearn.metrics import mean_absolute_error



In [2]:
dtypes = {
    "id": "int32",
    "breath_id": "int32",
    "R": "int16",
    "C": "int16",
    "time_step": "float32",
    "u_in": "float32",
    "u_out": "int8",
    "pressure": "float32",
}
df_train = pd.read_csv(
    "../input/ventilator-pressure-prediction/train.csv", dtype=dtypes
)
df_test = pd.read_csv("../input/ventilator-pressure-prediction/test.csv", dtype=dtypes)

df_train["breath_id"] = df_train["breath_id"].astype("category")
df_test["breath_id"] = df_test["breath_id"].astype("category")

df_train["RC"] = df_train["R"].astype(np.float32) * df_train["C"].astype(np.float32)
df_test["RC"] = df_test["R"].astype(np.float32) * df_test["C"].astype(np.float32)

df_train["u_in_sq"] = df_train["u_in"] ** 2
df_test["u_in_sq"] = df_test["u_in"] ** 2

grp_train = df_train.groupby("breath_id", sort=False)
grp_test = df_test.groupby("breath_id", sort=False)

df_train["u_in_cumsum"] = grp_train["u_in"].cumsum().astype(np.float32)
df_test["u_in_cumsum"] = grp_test["u_in"].cumsum().astype(np.float32)

df_train["u_in_lag1"] = grp_train["u_in"].shift(1).fillna(0).astype(np.float32)
df_test["u_in_lag1"] = grp_test["u_in"].shift(1).fillna(0).astype(np.float32)

df_train["u_in_diff"] = grp_train["u_in"].diff().fillna(0).astype(np.float32)
df_test["u_in_diff"] = grp_test["u_in"].diff().fillna(0).astype(np.float32)

df_train["breath_len"] = grp_train["id"].transform("size").astype(np.float32)
df_test["breath_len"] = grp_test["id"].transform("size").astype(np.float32)

df_train["u_in_time"] = (df_train["u_in"] * df_train["time_step"]).astype(np.float32)
df_test["u_in_time"] = (df_test["u_in"] * df_test["time_step"]).astype(np.float32)

df_train["R_div_C"] = df_train["R"].astype(np.float32) / (
    df_train["C"].astype(np.float32) + 1e-6
)
df_test["R_div_C"] = df_test["R"].astype(np.float32) / (
    df_test["C"].astype(np.float32) + 1e-6
)

df_train["C_div_R"] = df_train["C"].astype(np.float32) / (
    df_train["R"].astype(np.float32) + 1e-6
)
df_test["C_div_R"] = df_test["C"].astype(np.float32) / (
    df_test["R"].astype(np.float32) + 1e-6
)

df_train["u_in_out"] = (df_train["u_in"] * df_train["u_out"]).astype(np.float32)
df_test["u_in_out"] = (df_test["u_in"] * df_test["u_out"]).astype(np.float32)

df_train["time_step_sq"] = (df_train["time_step"] ** 2).astype(np.float32)
df_test["time_step_sq"] = (df_test["time_step"] ** 2).astype(np.float32)

df_train["R_u_in"] = (df_train["R"].astype(np.float32) * df_train["u_in"]).astype(
    np.float32
)
df_test["R_u_in"] = (df_test["R"].astype(np.float32) * df_test["u_in"]).astype(
    np.float32
)

df_train["C_u_in"] = (df_train["C"].astype(np.float32) * df_train["u_in"]).astype(
    np.float32
)
df_test["C_u_in"] = (df_test["C"].astype(np.float32) * df_test["u_in"]).astype(
    np.float32
)

df_train["RC_u_in"] = (df_train["RC"] * df_train["u_in"]).astype(np.float32)
df_test["RC_u_in"] = (df_test["RC"] * df_test["u_in"]).astype(np.float32)

unique_pressures = df_train["pressure"].unique()
sorted_pressures = np.sort(unique_pressures).astype(np.float32)
total_pressures_len = len(sorted_pressures)


def nearest_array(preds):
    """Vectorized nearest‑pressure lookup for a NumPy array."""
    preds = np.asarray(preds, dtype=np.float32)
    idx = np.searchsorted(sorted_pressures, preds, side="left")
    idx = np.clip(idx, 0, total_pressures_len - 1)

    lower_idx = np.maximum(idx - 1, 0)

    lower_val = sorted_pressures[lower_idx]
    upper_val = sorted_pressures[idx]

    lower_diff = np.abs(preds - lower_val)
    upper_diff = np.abs(upper_val - preds)

    choose_upper = upper_diff < lower_diff
    return np.where(choose_upper, upper_val, lower_val)


def set_seed(seed=2021):
    np.random.seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)




/tmp/ipykernel_11/3352727829.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grp_train = df_train.groupby("breath_id", sort=False)
/tmp/ipykernel_11/3352727829.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grp_test = df_test.groupby("breath_id", sort=False)


In [3]:
def wc(input_list):
    l = []
    for i in range(len(input_list)):
        public_lb_score = int(input_list[i].split("/")[-1].split(".")[1].split(" ")[0])
        l.append(public_lb_score)
        input_list[i] = (pd.read_csv(input_list[i]).pressure).ravel()
    output = 0
    l_sum = sum(l)
    if len(input_list) == 1:
        output = input_list[0]
    else:
        weight1 = (l[1] / l_sum) + 0.1
        weight2 = 1 - weight1
        output += input_list[0] * weight1 + input_list[1] * weight2
    return output


def blend(a_path, b_path):
    """Blend two existing submission files if they exist; otherwise noop."""
    if os.path.exists(a_path) and os.path.exists(b_path):
        a = pd.read_csv(a_path, dtype={"id": "int32", "pressure": "float32"})
        b = pd.read_csv(b_path, dtype={"id": "int32", "pressure": "float32"})
        a.pressure = a.pressure * 0.65 + b.pressure * 0.35
        a["pressure"] = nearest_array(a.pressure.values)
        a.to_csv("blend.csv", index=False)
        return a
    else:
        print("Blend files not found; skipping blending.")
        return None




In [4]:
def train_and_predict():
    set_seed(2021)

    # Feature columns: everything except target and id (same as original)
    feature_cols = [c for c in df_train.columns if c not in ["pressure", "id"]]

    # Convert once to NumPy (float32) – same as original
    X = df_train[feature_cols].to_numpy(dtype=np.float32, copy=False)
    y = df_train["pressure"].to_numpy(dtype=np.float32, copy=False)

    # ------------------------------------------------------------------
    # Speed optimisation: use a small random hold‑out only for reporting.
    # The model will still perform its own internal validation via early_stopping.
    # ------------------------------------------------------------------
    holdout_size = int(0.05 * len(y))  # 5 % of data
    holdout_idx = np.random.choice(len(y), size=holdout_size, replace=False)
    X_val = X[holdout_idx]
    y_val = y[holdout_idx]

    # Reduce max_iter – early_stopping will terminate long before the limit.
    model = HistGradientBoostingRegressor(
        loss="absolute_error",
        max_iter=500,  # lowered from 2500; early stopping controls actual iterations
        learning_rate=0.005,
        max_depth=15,
        l2_regularization=0.1,
        random_state=42,
        early_stopping=True,
        n_iter_no_change=20,
        validation_fraction=0.1,
    )

    gc.collect()
    model.fit(X, y)  # fit on full training data; internal validation used automatically

    # Evaluate on the cheap hold‑out set for logging (identical metric calculation)
    val_pred = model.predict(X_val)
    val_mae_raw = mean_absolute_error(y_val, val_pred)
    val_pred_rounded = nearest_array(val_pred)
    val_mae_rounded = mean_absolute_error(y_val, val_pred_rounded)
    print(f"Validation MAE (raw): {val_mae_raw:.5f}")
    print(f"Validation MAE (rounded): {val_mae_rounded:.5f}")

    # Predict test set
    test_pred = model.predict(
        df_test[feature_cols].to_numpy(dtype=np.float32, copy=False)
    )
    test_pred_rounded = nearest_array(test_pred)

    submission = pd.DataFrame({"id": df_test["id"], "pressure": test_pred_rounded})
    submission_path = "submission.csv"
    submission.to_csv(submission_path, index=False)
    print(f"Submission written to {submission_path}")

    # Clean up
    del X, y, X_val, y_val, test_pred
    gc.collect()


train_and_predict()




Validation MAE (raw): 1.36977
Validation MAE (rounded): 1.36943
Submission written to submission.csv


In [5]:
a_path = "../input/gb-data-blending-recover/0.1371 blend.csv"
b_path = "../input/gb-data-blending-recover/0.1377 blend.csv"
blend(a_path, b_path)

Blend files not found; skipping blending.
